# Assignment 10 – Digital Image Processing

**Topics**
1. Image denoising using Median Filter and Adaptive Median Filter, with PSNR comparison.
2. Image segmentation using Otsu's Global Thresholding.

**Instructions:** Run the notebook from top to bottom. When prompted, enter the image path and filter window sizes. The notebook also supports uploading an image in Google Colab.

## 1. Import required libraries

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 2. Load the input image

Enter the path of your image below. In Google Colab, you can upload an image using the optional upload cell.
For this assignment, the denoising experiment uses a grayscale image.

In [ ]:
# If using Google Colab, run this cell to upload an image.
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        image_path = next(iter(uploaded))
    else:
        image_path = input("Enter image path: ").strip()
except ImportError:
    image_path = input("Enter image path (for example, image.jpg): ").strip()

image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if image is None:
    raise FileNotFoundError(f"Could not read image: {image_path}")

print("Image loaded successfully.")
print("Image dimensions:", image.shape)

plt.figure(figsize=(6, 5))
plt.imshow(image, cmap="gray")
plt.title("Original Grayscale Image")
plt.axis("off")
plt.show()

## 3. Add salt-and-pepper noise

To compare denoising filters quantitatively, we need a noisy image and a clean reference image. The original input image is treated as the clean reference. Salt-and-pepper noise is added to it.

The noise amount controls the proportion of pixels changed to black or white.

In [ ]:
def add_salt_pepper_noise(img, noise_amount=0.05):
    noisy = img.copy()
    total = img.size
    num_salt = int(total * noise_amount / 2)
    num_pepper = int(total * noise_amount / 2)

    # Random salt (white) pixels
    coords = (
        np.random.randint(0, img.shape[0], num_salt),
        np.random.randint(0, img.shape[1], num_salt)
    )
    noisy[coords] = 255

    # Random pepper (black) pixels
    coords = (
        np.random.randint(0, img.shape[0], num_pepper),
        np.random.randint(0, img.shape[1], num_pepper)
    )
    noisy[coords] = 0
    return noisy

np.random.seed(42)
noise_amount = 0.05
noisy_image = add_salt_pepper_noise(image, noise_amount)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(noisy_image, cmap="gray")
plt.title(f"Noisy Image (noise amount={noise_amount})")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Ordinary Median Filter

A median filter replaces each pixel with the median value of the pixels in its neighborhood. It is effective at removing salt-and-pepper noise.

The kernel size must be an odd positive integer.

In [ ]:
median_kernel = int(input("Enter median filter kernel size (odd number, e.g. 3 or 5): "))

if median_kernel < 1 or median_kernel % 2 == 0:
    raise ValueError("Kernel size must be a positive odd integer.")

median_filtered = cv2.medianBlur(noisy_image, median_kernel)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(noisy_image, cmap="gray")
plt.title("Noisy Image")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(median_filtered, cmap="gray")
plt.title(f"Median Filter ({median_kernel}x{median_kernel})")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(image, cmap="gray")
plt.title("Original Reference")
plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Adaptive Median Filter

Unlike a standard median filter, the adaptive median filter changes its neighborhood size according to local image characteristics. It starts with the initial window size and increases it until the pixel can be classified as either an impulse-noise pixel or a valid image pixel, or until the maximum window size \(S_{max}\) is reached.

For each neighborhood:
- **Stage A:** Compare the median with the minimum and maximum neighborhood values.
- **Stage B:** If the median is a valid image value, check whether the centre pixel is valid. If it is not, replace it with the median.
- If Stage A cannot find a suitable median, increase the window size.
- If the maximum size is reached, use the median as the replacement.

Both the initial and maximum window sizes must be odd, and the maximum must be at least as large as the initial size.

In [ ]:
def adaptive_median_filter(img, initial_size=3, smax=7):
    if initial_size < 3 or initial_size % 2 == 0:
        raise ValueError("Initial window size must be an odd integer >= 3.")
    if smax % 2 == 0 or smax < initial_size:
        raise ValueError("Smax must be an odd integer >= initial window size.")

    padded = np.pad(img, smax // 2, mode="edge")
    output = img.copy()
    rows, cols = img.shape

    for i in range(rows):
        for j in range(cols):
            window_size = initial_size
            pixel = int(img[i, j])
            replacement = pixel

            while True:
                radius = window_size // 2
                window = padded[
                    i + smax // 2 - radius : i + smax // 2 + radius + 1,
                    j + smax // 2 - radius : j + smax // 2 + radius + 1
                ]

                z_min = int(np.min(window))
                z_max = int(np.max(window))
                z_med = int(np.median(window))

                # Stage A
                if z_min < z_med < z_max:
                    # Stage B
                    if z_min < pixel < z_max:
                        replacement = pixel
                    else:
                        replacement = z_med
                    break
                else:
                    window_size += 2
                    if window_size > smax:
                        replacement = z_med
                        break

            output[i, j] = replacement

    return output


initial_size = int(input("Enter initial adaptive window size (odd, >=3): "))
smax = int(input("Enter maximum window size Smax (odd and >= initial size): "))

adaptive_filtered = adaptive_median_filter(noisy_image, initial_size, smax)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(noisy_image, cmap="gray")
plt.title("Noisy Image")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(adaptive_filtered, cmap="gray")
plt.title(f"Adaptive Median ({initial_size} to {smax})")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(image, cmap="gray")
plt.title("Original Reference")
plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Calculate PSNR and compare the filters

Peak Signal-to-Noise Ratio (PSNR) measures the similarity between a restored image and the clean reference image. A higher PSNR generally indicates less reconstruction error.

\[
MSE = \frac{1}{MN}\sum_{i=1}^{M}\sum_{j=1}^{N}(I(i,j)-K(i,j))^2
\]

\[
PSNR = 10\log_{10}\left(\frac{MAX^2}{MSE}\right)
\]

Here, \(MAX=255\) for an 8-bit grayscale image. If MSE is zero, PSNR is infinite.

In [ ]:
def calculate_psnr(reference, restored):
    reference = reference.astype(np.float64)
    restored = restored.astype(np.float64)
    mse = np.mean((reference - restored) ** 2)

    if mse == 0:
        return float("inf")

    max_pixel = 255.0
    return 10 * np.log10((max_pixel ** 2) / mse)


psnr_noisy = calculate_psnr(image, noisy_image)
psnr_median = calculate_psnr(image, median_filtered)
psnr_adaptive = calculate_psnr(image, adaptive_filtered)

print(f"PSNR of noisy image          : {psnr_noisy:.2f} dB")
print(f"PSNR after median filtering  : {psnr_median:.2f} dB")
print(f"PSNR after adaptive filtering: {psnr_adaptive:.2f} dB")

print("\nComparison:")
if psnr_adaptive > psnr_median:
    print("Adaptive median filter gives a higher PSNR for this experiment.")
elif psnr_adaptive < psnr_median:
    print("Ordinary median filter gives a higher PSNR for this experiment.")
else:
    print("Both filters give the same PSNR for this experiment.")

## 7. Visual comparison of denoising results

In [ ]:
images = [image, noisy_image, median_filtered, adaptive_filtered]
titles = [
    "Original",
    "Noisy",
    f"Median Filter ({median_kernel}x{median_kernel})",
    f"Adaptive Median ({initial_size} to {smax})"
]

plt.figure(figsize=(14, 8))
for index, (img, title) in enumerate(zip(images, titles), start=1):
    plt.subplot(2, 2, index)
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Otsu's Global Thresholding for Image Segmentation

Otsu's method automatically determines a global threshold from the grayscale image histogram. It selects a threshold that separates pixels into two classes (background and foreground) by maximizing the between-class variance.

The method does not require the threshold value to be manually specified.

In [ ]:
# Otsu's thresholding requires a grayscale image.
otsu_input = image.copy()

threshold_value, otsu_binary = cv2.threshold(
    otsu_input,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Otsu threshold value:", threshold_value)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(otsu_input, cmap="gray")
plt.title("Input Grayscale Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.hist(otsu_input.ravel(), bins=256, range=(0, 256))
plt.axvline(threshold_value, linestyle="--", label=f"Threshold = {threshold_value:.0f}")
plt.title("Grayscale Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()

plt.subplot(1, 3, 3)
plt.imshow(otsu_binary, cmap="gray")
plt.title("Otsu Binary Segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

## 9. Otsu's Thresholding with Inverted Output (Optional)

If the foreground and background appear reversed for your image, use `THRESH_BINARY_INV` instead. This does not change how Otsu calculates the threshold; it reverses the binary output.

In [ ]:
_, otsu_binary_inv = cv2.threshold(
    otsu_input,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(otsu_binary, cmap="gray")
plt.title("Otsu Binary")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(otsu_binary_inv, cmap="gray")
plt.title("Otsu Binary Inverted")
plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Result and Conclusion

**Experiment 1:** Salt-and-pepper noise was added to the grayscale input image. The ordinary median filter and adaptive median filter were applied. Their denoising performance was compared using PSNR values. The filter with the higher PSNR produced a result closer to the original reference image for this experiment.

**Experiment 2:** Otsu's global thresholding method was applied to the grayscale image. The algorithm automatically selected a threshold and generated a binary segmented image.

**Note:** The PSNR comparison depends on the selected image, noise amount, and filter window sizes. Record the actual PSNR values printed when you run the notebook.